# Self-Correcting Code Agent — Core Loop Prototype

Prototype of the ReAct + Reflexion loop described in `ReAct_Reflexion_Project_Overview.md`.

**What's implemented here:**
- Reasoning/Planning turn via a **Groq-hosted Qwen model**, configured through an env var (`MODEL_ID`) rather than hardcoded, since the exact model slug is expected to change
- A **`subprocess`-based execution sandbox** (real process isolation, not a restricted `exec()` namespace)
- Reflexion turn on failure, appended to the same conversation thread
- A **3-attempt cap** and a **human-in-the-loop confirmation** checkpoint after any successful run
- A trajectory logger for the demo/report
- A **vanilla HTML/CSS/JS UI** in `frontend/` (host on GitHub Pages) with a **Settings panel** to point it at this notebook's backend over a **Cloudflare tunnel**. Auto-growing task box + example chips, live NDJSON trajectory with the full sandbox result (stdout / stderr / exit code / duration) shown per attempt, and clickable **Yes / No** confirmation buttons.

Run the cells top to bottom. The API key is read from a Colab secret (Section 2) — you set it once.

## 1. Install dependencies

In [4]:
!pip install -q openai flask flask-cors

## 2. Configure the LLM backend

**API key comes from a Colab secret — you set it once, never re-type it.**

1. Click the **key icon** (🔑) in the Colab left sidebar.
2. Add a secret named **`GROQ_API_KEY`**, paste your key from [console.groq.com/keys](https://console.groq.com/keys).
3. Toggle **Notebook access** on for it.

The model id is read from the `MODEL_ID` env var — **not hardcoded** — since the team expects to swap the exact model slug over time (current: `qwen/qwen3.8-27b`). Verify it against [console.groq.com/docs/models](https://console.groq.com/docs/models).

**Groq free tier caps output tokens/minute (OTPM ~1000).** Two defences: every call sets `max_tokens=MAX_OUTPUT_TOKENS` (default 800, env-overridable) so Groq doesn't reject it up front for a large *expected* output; and `call_llm` retries on HTTP 429, honoring the `Retry-After` header. A multi-attempt task may still pause ~30–60s between calls on this tier — that's the rate limit, not a hang. Set `MODEL_ID` to a higher-limit model (e.g. `llama-3.3-70b-versatile`) if that's too slow.

In [5]:
import os, time

api_key = None
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("GROQ_API_KEY")
except Exception:
    api_key = os.environ.get("GROQ_API_KEY")  # non-Colab fallback: plain env var

if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY not found. In Colab: key icon in the left sidebar -> add secret "
        "named GROQ_API_KEY -> enable 'Notebook access'. Then re-run this cell."
    )

# Both read from the environment, not hardcoded into the API call below.
# setdefault() only fills in a value if nothing else (shell env, Colab secret, prior cell) set it.
os.environ.setdefault("MODEL_ID", "qwen/qwen3.8-27b")
os.environ.setdefault("MAX_OUTPUT_TOKENS", "800")  # keep under Groq free-tier OTPM (~1000)
MODEL_ID = os.environ["MODEL_ID"]
MAX_OUTPUT_TOKENS = int(os.environ["MAX_OUTPUT_TOKENS"])

from openai import OpenAI
from openai import RateLimitError

BASE_URL = "https://api.groq.com/openai/v1"
client = OpenAI(base_url=BASE_URL, api_key=api_key, max_retries=0)  # we handle retries ourselves

def call_llm(messages, temperature=0.2, max_retries=5):
    """One call in the task's ongoing thread (planning or reflection).
    Caps output tokens and backs off on 429s so the Groq free-tier OTPM limit
    slows the loop down instead of crashing it."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=messages,
                temperature=temperature,
                max_tokens=MAX_OUTPUT_TOKENS,
            )
            return resp.choices[0].message.content
        except RateLimitError as e:
            if attempt == max_retries:
                raise
            wait = 30.0
            try:
                ra = e.response.headers.get("retry-after")
                if ra:
                    wait = float(ra) + 1
            except Exception:
                pass
            print(f"[groq rate limit; waiting {wait:.0f}s then retrying ({attempt + 1}/{max_retries})]")
            time.sleep(wait)

## 3. System prompt

Defines the ReAct output format: a short *Thought*, then exactly one fenced Python code block. The code must be a complete, runnable script — whatever it prints is what becomes the Observation. It also tells the model how to use **user-supplied input** (§6a): read it from stdin rather than inventing example data.

In [6]:
SYSTEM_PROMPT = """You are a careful Python coding agent working inside a ReAct (Reason+Act) loop.

For every turn:
1. Write a short \"Thought:\" section explaining your plan or, on a retry, how you're addressing the previous failure/reflection.
2. Then write exactly ONE fenced python code block containing a complete, runnable script that accomplishes the task.
   - The script must PRINT whatever output demonstrates the result (stdout is the only thing the agent observes).
   - If an \"Input\" block is given below the task, it is piped to the script's stdin (not typed live, so it's
     always safe to read). Read it with sys.stdin.read() or input() and parse it however the task requires
     (numbers, a Python-literal list, CSV, freeform text, ...) -- use that exact data, never invent your own.
   - If no Input block is given, do not call input()/stdin at all -- construct any example data the script
     needs directly in the code.
   - Do not access the network or the filesystem outside the current working directory.
3. Do not include more than one code block.

You will be shown the Observation (stdout/stderr/traceback) after each attempt, and may be asked to reflect on a failure before your next attempt."""

REFLECTION_PROMPT = (
    "That attempt did not solve the task. Diagnose the root cause of the failure as specifically as you can "
    "(not just what error occurred, but why the code produced it), and state concretely what you will change "
    "in the next attempt. Do not write code yet -- just the diagnosis and plan."
)

REJECTED_BY_USER_PROMPT = (
    "The code ran without error, but the user reviewed the output and confirmed it does NOT correctly solve the task. "
    + REFLECTION_PROMPT
)

## 4. Code extractor

Pulls the fenced code block out of the model's response and validates it parses before we ever try to run it.

In [7]:
import re
import ast

CODE_BLOCK_RE = re.compile(r"```(?:python)?\s*\n(.*?)```", re.DOTALL)

def extract_code(response_text):
    """Returns (code, error). Exactly one of the two is None."""
    match = CODE_BLOCK_RE.search(response_text)
    if not match:
        return None, "No fenced python code block found in the response."
    code_str = match.group(1).strip()
    try:
        ast.parse(code_str)
    except SyntaxError as e:
        return None, f"Extracted code has a syntax error: {e}"
    return code_str, None

def extract_thought(response_text):
    idx = response_text.find("```")
    thought = response_text[:idx].strip() if idx != -1 else response_text.strip()
    return thought or "(no explicit reasoning provided)"

## 5. Execution sandbox (`subprocess`-based)

Each attempt runs as a **separate OS process** in its own interpreter (`sys.executable -I`), not `exec()` in the orchestrator's namespace. That's a real isolation boundary: the attempt cannot see or mutate the orchestrator's memory, and a fresh process per attempt means no state can leak between attempts.

**Documented limitation (matches §8/§11 of the design doc):** this isolates the *process*, but does not fully sandbox *system calls* — e.g. it doesn't block outbound network access or restrict which stdlib modules can be imported. A production version would add OS-level controls (containers, seccomp, `resource.setrlimit`, an egress-blocking network namespace). For this prototype we rely on: a fresh scratch directory per attempt, a timeout, a minimal environment, and truncated output capture.

In [8]:
import subprocess
import sys
import tempfile
import os
import shutil
import time

SANDBOX_TIMEOUT = 10  # seconds

def run_in_sandbox(code_str, timeout=SANDBOX_TIMEOUT, max_output_chars=4000, stdin_data=""):
    scratch_dir = tempfile.mkdtemp(prefix="agent_attempt_")
    script_path = os.path.join(scratch_dir, "attempt.py")
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(code_str)

    # Minimal environment: no inherited vars beyond PATH, so the child can't read
    # the orchestrator's secrets (e.g. GROQ_API_KEY) out of os.environ.
    env = {"PATH": os.environ.get("PATH", ""), "PYTHONIOENCODING": "utf-8"}
    argv = [sys.executable, "-I", script_path]  # -I = isolated mode (ignore env/user site)

    timed_out = False
    started = time.monotonic()
    try:
        proc = subprocess.run(
            argv,
            cwd=scratch_dir,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=env,
            input=stdin_data,  # "" still closes stdin at EOF, so a stray input() fails fast instead of hanging
        )
        stdout, stderr, returncode = proc.stdout, proc.stderr, proc.returncode
    except subprocess.TimeoutExpired as e:
        timed_out = True
        stdout = e.stdout or ""
        stderr = (e.stderr or "") + f"\n[Timed out after {timeout}s]"
        returncode = None
    finally:
        duration_s = round(time.monotonic() - started, 3)
        shutil.rmtree(scratch_dir, ignore_errors=True)

    def trunc(s):
        return s if len(s) <= max_output_chars else s[:max_output_chars] + "\n...[truncated]"

    success = (not timed_out) and returncode == 0
    return {
        "success": success,
        "stdout": trunc(stdout),
        "stderr": trunc(stderr),
        "returncode": returncode,
        "timed_out": timed_out,
        "duration_s": duration_s,
        "timeout_s": timeout,
        "isolation": "subprocess (python -I), fresh cwd, minimal env",
        "argv": " ".join(argv[:2]) + " <script>",
    }

def format_observation(result):
    lines = []
    if result["timed_out"]:
        lines.append("Execution timed out.")
    lines.append(f"Return code: {result['returncode']}  (ran {result['duration_s']}s in {result['isolation']})")
    if result["stdout"]:
        lines.append("stdout:\n" + result["stdout"])
    if result["stderr"]:
        lines.append("stderr/traceback:\n" + result["stderr"])
    if result["success"]:
        lines.append("[No error -- execution succeeded]")
    return "\n\n".join(lines)

## 6. Trajectory logger

Every (thought, code, observation, reflection) turn across every task gets appended here, independent of the LLM thread itself, so it's easy to render for the report.

In [9]:
TRAJECTORY_LOG = []  # list of dicts, one per attempt across all tasks/sessions

def render_entry_md(entry):
    parts = [f"### Attempt {entry['attempt']}", "", "**Thought / Plan:**", "", entry["thought"], ""]
    if entry.get("code"):
        parts += ["**Action (code):**", "```python", entry["code"], "```", ""]
    parts += ["**Observation:**", "```", entry["observation"], "```", ""]
    return "\n".join(parts)

def render_reflection_md(reflection):
    return "\n".join(["**Reflection:**", "", reflection, "", "---", ""])

## 7. Agent session — the ReAct + Reflexion loop controller

A small state machine so the same loop logic drives both the headless notebook demo and the front-end:

- `status == "pending"` — ready for another plan+execute turn
- `step()` — one Thought -> Action -> Observation turn; on failure with attempts remaining, immediately runs the Reflexion turn in the same thread and goes back to `"pending"`; on failure with no attempts left, goes to `"stopped"`; on success, goes to `"awaiting_confirmation"`
- `confirm(accepted)` — the HITL checkpoint after a successful run; `True` -> `"succeeded"`, `False` -> back into the loop (as a Reflexion-worthy failure) if attempts remain, else `"stopped"`

This mirrors §5/§10 of the design doc: one continuous LLM thread per task, 3-attempt cap, HITL success check. Each session gets a `session_id` so the Flask backend in §10 can look it up across separate HTTP requests.

In [10]:
import uuid

class AgentSession:
    def __init__(self, task, max_iterations=3, input_data=""):
        self.session_id = str(uuid.uuid4())
        self.task = task
        self.input_data = input_data or ""
        self.max_iterations = max_iterations
        self.attempt = 0
        self.status = "pending"  # pending | awaiting_confirmation | succeeded | stopped
        self.last_code = None
        self.last_output = None
        self.trajectory = []
        task_block = f"## Task\n\n{task}\n"
        if self.input_data:
            task_block += f"\n## Input (piped to stdin)\n\n```\n{self.input_data}\n```\n"
        self.trajectory_md = task_block + "\n---\n\n"
        user_msg = f"Task: {task}"
        if self.input_data:
            user_msg += f"\n\nInput (piped to stdin when the code runs):\n{self.input_data}"
        self.messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]

    def step(self):
        assert self.status == "pending", f"step() called while status={self.status!r}"
        self.attempt += 1

        plan_response = call_llm(self.messages)
        self.messages.append({"role": "assistant", "content": plan_response})

        thought = extract_thought(plan_response)
        code_str, extract_err = extract_code(plan_response)

        if code_str is None:
            observation = f"[Code extraction failed] {extract_err}"
            result = None
        else:
            result = run_in_sandbox(code_str, stdin_data=self.input_data)
            observation = format_observation(result)

        self.messages.append({"role": "user", "content": f"Observation:\n{observation}"})

        entry = {
            "task": self.task,
            "attempt": self.attempt,
            "thought": thought,
            "code": code_str,
            "extract_error": extract_err,
            "sandbox": result,  # structured: stdout, stderr, returncode, timed_out, duration_s, ... (None if no code ran)
            "observation": observation,
            "success": bool(result and result["success"]),
            "error_type": None if (result and result["success"]) else _classify_failure(code_str, result, extract_err),
            "reflection": None,
        }
        self.trajectory.append(entry)
        TRAJECTORY_LOG.append(entry)
        self.trajectory_md += render_entry_md(entry)

        if result is not None and result["success"]:
            self.status = "awaiting_confirmation"
            self.last_code = code_str
            self.last_output = result["stdout"]
        elif self.attempt >= self.max_iterations:
            self.status = "stopped"
        else:
            reflection = call_llm(self.messages + [{"role": "user", "content": REFLECTION_PROMPT}])
            self.messages.append({"role": "user", "content": REFLECTION_PROMPT})
            self.messages.append({"role": "assistant", "content": reflection})
            entry["reflection"] = reflection
            self.trajectory_md += render_reflection_md(reflection)
            self.status = "pending"

        return entry

    def confirm(self, accepted):
        assert self.status == "awaiting_confirmation", f"confirm() called while status={self.status!r}"
        if accepted:
            self.status = "succeeded"
            self.trajectory_md += "**User confirmed: this solves the task.**\n\n---\n\n"
            return

        self.trajectory_md += "**User rejected: output does not solve the task.**\n\n"
        if self.attempt >= self.max_iterations:
            self.status = "stopped"
            self.trajectory_md += "---\n\n"
            return

        self.messages.append({"role": "user", "content": REJECTED_BY_USER_PROMPT})
        reflection = call_llm(self.messages)
        self.messages.append({"role": "assistant", "content": reflection})
        if self.trajectory:
            self.trajectory[-1]["reflection"] = reflection
            self.trajectory[-1]["success"] = False
            self.trajectory[-1]["error_type"] = "rejected_by_user"
        self.trajectory_md += render_reflection_md(reflection)
        self.status = "pending"

### 7a. Repeated-failure classification

Per §9 of the design doc: two failed attempts count as the *same error type* if they share both the exception class and the failing line. This is logged automatically per attempt rather than judged by eye, so `TRAJECTORY_LOG` supports the ReAct-only vs. ReAct+Reflexion ablation later.

In [11]:
import re as _re

_TRACEBACK_LAST_LINE_RE = _re.compile(r"^(\w+(?:Error|Exception|Warning))\b", _re.MULTILINE)
_FAILING_LINE_RE = _re.compile(r'File "[^"]*", line (\d+)')

def _classify_failure(code_str, result, extract_err):
    """Returns a compact (exception_class, failing_line) tag used to detect repeated identical failures."""
    if extract_err is not None:
        return f"extraction_error:{extract_err.split(':')[0]}"
    if result is None:
        return "unknown_error"
    if result["timed_out"]:
        return "timeout"
    stderr = result["stderr"] or ""
    exc_matches = _TRACEBACK_LAST_LINE_RE.findall(stderr)
    exc_class = exc_matches[-1] if exc_matches else f"exit_code_{result['returncode']}"
    line_matches = _FAILING_LINE_RE.findall(stderr)
    failing_line = line_matches[-1] if line_matches else "?"
    return f"{exc_class}@line_{failing_line}"

## 8. Headless demo (runs in the notebook, HITL via `input()`)

Drives an `AgentSession` to completion, printing each Thought/Action/Observation/Reflection as it happens, and pausing for a real y/n confirmation once the code runs cleanly.

In [12]:
def _drive(session):
    """Generator: advances the session one step() at a time while status == 'pending'."""
    while session.status == "pending":
        session.step()
        yield session

def run_headless(task, max_iterations=3, input_data=""):
    session = AgentSession(task, max_iterations=max_iterations, input_data=input_data)
    for _ in _drive(session):
        pass
    print(session.trajectory_md)

    while session.status == "awaiting_confirmation":
        print(f"--- Code ran without error. Output:\n{session.last_output}\n")
        answer = input("Does this correctly solve the task? [y/n]: ").strip().lower()
        session.confirm(answer.startswith("y"))
        for _ in _drive(session):
            pass
        print(session.trajectory_md)

    print(f"Final status: {session.status} (after {session.attempt} attempt(s))")
    return session

In [13]:
# Try it: an "easy" task per the design doc's evaluation plan (simple data transformation)
demo_session = run_headless(
    "Write a function that takes a list of numbers and returns a new list with duplicates removed, "
    "preserving the original order. Print the result for [4, 5, 4, 2, 5, 1, 2]."
)

## Task

Write a function that takes a list of numbers and returns a new list with duplicates removed, preserving the original order. Print the result for [4, 5, 4, 2, 5, 1, 2].

---

### Attempt 1

**Thought / Plan:**

(no explicit reasoning provided)

**Action (code):**
```python
def remove_duplicates(lst):
    seen = set()
    result = []
    for num in lst:
        if num not in seen:
            seen.add(num)
            result.append(num)
    return result

# Test with the given list
test_list = [4, 5, 4, 2, 5, 1, 2]
result = remove_duplicates(test_list)
print(result)
```

**Observation:**
```
Return code: 0  (ran 0.079s in subprocess (python -I), fresh cwd, minimal env)

stdout:
[4, 5, 2, 1]


[No error -- execution succeeded]
```

--- Code ran without error. Output:
[4, 5, 2, 1]


Does this correctly solve the task? [y/n]: y
## Task

Write a function that takes a list of numbers and returns a new list with duplicates removed, preserving the original order. Print the result for 

### 8a. Same thing, with your own input data

`input_data` is piped to the script's stdin — the model is told (via the system prompt) to read it instead of inventing example values. This is what "sort my own list" looks like: describe the task generically, then supply the actual data separately.

In [14]:
# Try it: task description stays generic, the data comes from input_data instead of being hardcoded.
demo_session_2 = run_headless(
    "Read a list of numbers from stdin (one per line, or comma-separated) and print it sorted in "
    "descending order.",
    input_data="8, 3, 21, 3, 1, 14, 5",
)

## Task

Read a list of numbers from stdin (one per line, or comma-separated) and print it sorted in descending order.

## Input (piped to stdin)

```
8, 3, 21, 3, 1, 14, 5
```

---

### Attempt 1

**Thought / Plan:**

Thought: I'll read stdin, parse the numbers (handling both comma-separated and line-separated formats), sort them in descending order, and print them.

**Action (code):**
```python
import sys

data = sys.stdin.read()
# Split by commas and/or newlines
parts = []
for line in data.splitlines():
    for token in line.split(','):
        token = token.strip()
        if token:
            parts.append(token)

numbers = [float(x) if ('.' in x) else int(x) for x in parts]
numbers.sort(reverse=True)

# Print as integers if all are ints
if all(isinstance(n, int) for n in numbers):
    print(', '.join(str(n) for n in numbers))
else:
    print(', '.join(str(n) for n in numbers))
```

**Observation:**
```
Return code: 0  (ran 0.024s in subprocess (python -I), fresh cwd, minimal env)

## 9. Trajectory report

Renders `TRAJECTORY_LOG` as a table for inspection/inclusion in the writeup.

In [15]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(TRAJECTORY_LOG)[["task", "attempt", "success", "error_type"]]

,task,attempt,success,error_type
0,Write a function that takes a list of numbers and returns a new list with du...,1,True,None
1,"Read a list of numbers from stdin (one per line, or comma-separated) and pri...",1,True,None


## 10. Backend API + Cloudflare tunnel

The UI is a static **vanilla HTML/CSS/JS** app in the repo's `frontend/` folder — you host it on **GitHub Pages** (or open it locally). It has a **Settings** panel where you paste the Cloudflare URL this notebook prints, and it stores that in `localStorage` so you set it once per browser.

This notebook is just the **backend**: a Flask API (`/api/health`, `/api/run`, `/api/confirm`) driving the same `AgentSession`/`_drive()` loop as the headless demo, with CORS enabled so the GitHub Pages origin can call it, exposed via a Cloudflare quick tunnel.

### 10a. Backend (Flask + CORS)

`CORS(app)` lets the cross-origin GitHub Pages page call `/api/*`. `/api/health` is what the Settings panel pings to verify the link. Each attempt in the streamed response now carries the **structured sandbox result** (stdout, stderr, exit code, duration, isolation) so the UI can show exactly what ran and what came back. `SESSIONS` is an in-memory dict — fine for a single-user demo (see limitations).

In [16]:
import json as _json, os
from flask import Flask, request, jsonify, Response, send_from_directory
from flask_cors import CORS

# Optional: if a local frontend/ folder is present, the tunnel serves the UI too
# (handy for local testing). Hosting on GitHub Pages instead? Then only the API matters.
FRONTEND_DIR = next(
    (os.path.abspath(p) for p in ("frontend", "/content/genai_lab2/frontend")
     if os.path.isfile(os.path.join(p, "index.html"))),
    None,
)

SESSIONS = {}
app = Flask(__name__, static_folder=None)
CORS(app)  # allow any origin to hit /api/* (GitHub Pages, localhost, ...)

def session_to_json(session):
    return {
        "session_id": session.session_id,
        "status": session.status,
        "attempt": session.attempt,
        "max_iterations": session.max_iterations,
        "status_text": render_status(session),
        "trajectory": session.trajectory,
        "last_output": session.last_output,
        "input_data": session.input_data,
    }

def render_status(session):
    if session.status == "succeeded":
        return f"Succeeded in {session.attempt} attempt(s)."
    if session.status == "stopped":
        return f"Stopped after {session.attempt} attempt(s) -- task not solved within the {session.max_iterations}-attempt budget."
    if session.status == "awaiting_confirmation":
        return f"Attempt {session.attempt} of {session.max_iterations} -- code ran clean. Confirm it solves the task, or reject to retry."
    return f"Attempt {session.attempt} of {session.max_iterations} -- retrying after failure..."

def _health_payload():
    # api_key is guaranteed truthy here -- Section 2 raises before this cell can even run
    # if GROQ_API_KEY was missing. Reported under a few likely field names since we can't
    # see the frontend's exact check.
    key_set = bool(api_key)
    return {
        "status": "ok",
        "model": MODEL_ID,
        "sandbox_timeout_s": SANDBOX_TIMEOUT,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "api_key_set": key_set,
        "has_api_key": key_set,
        "groq_api_key_set": key_set,
    }

@app.route("/api/health")
def api_health():
    return jsonify(_health_payload())

@app.route("/health")  # alias -- some frontends check the short path instead of /api/health
def health_alias():
    return jsonify(_health_payload())

@app.route("/api/run", methods=["POST"])
def api_run():
    data = request.get_json(force=True) or {}
    task = (data.get("task") or "").strip()
    if not task:
        return jsonify({"error": "task is required"}), 400

    session = AgentSession(
        task,
        max_iterations=int(data.get("max_iterations") or 3),
        input_data=data.get("input_data") or "",
    )
    SESSIONS[session.session_id] = session

    def generate():
        for _ in _drive(session):
            yield _json.dumps(session_to_json(session)) + "\n"
        yield _json.dumps(session_to_json(session)) + "\n"

    return Response(generate(), mimetype="application/x-ndjson")

@app.route("/api/confirm", methods=["POST"])
def api_confirm():
    data = request.get_json(force=True) or {}
    session = SESSIONS.get(data.get("session_id"))
    if session is None or session.status != "awaiting_confirmation":
        return jsonify({"error": "no active run awaiting confirmation"}), 400
    accepted = bool(data.get("accepted"))

    def generate():
        session.confirm(accepted)
        yield _json.dumps(session_to_json(session)) + "\n"
        for _ in _drive(session):
            yield _json.dumps(session_to_json(session)) + "\n"
        yield _json.dumps(session_to_json(session)) + "\n"

    return Response(generate(), mimetype="application/x-ndjson")

if FRONTEND_DIR:
    @app.route("/")
    def index():
        return send_from_directory(FRONTEND_DIR, "index.html")

    @app.route("/<path:filename>")
    def static_files(filename):
        return send_from_directory(FRONTEND_DIR, filename)
    print("Tunnel will also serve the local UI from:", FRONTEND_DIR)
else:
    @app.route("/")
    def _status_page():
        # so opening the tunnel URL directly shows something useful instead of a bare 404
        return jsonify({
            **_health_payload(),
            "message": "Self-Correcting Code Agent backend is running. Point your frontend's backend URL here.",
            "endpoints": ["/api/health", "/api/run", "/api/confirm"],
        })
    print("No local frontend/ -- API only. Use your GitHub Pages UI and paste the tunnel URL into its Settings.")

No local frontend/ -- API only. Use your GitHub Pages UI and paste the tunnel URL into its Settings.


### 10b. Launch + Cloudflare tunnel

Starts the Flask API in a background thread, opens a **Cloudflare quick tunnel** (no account needed — the binary downloads on first run), and prints the public `https://<random>.trycloudflare.com` URL.

**This cell blocks on purpose — leave it running.** The backend is up only while this cell shows as running *and* the Colab runtime is connected. Colab's free tier disconnects after ~90 min idle (12 h max), which is why "it ran all the cells but nothing responds" happens — the runtime died and took Flask + the tunnel with it.

**Each time you (re)start this cell, copy the printed URL into the frontend's “Connect the backend” box** — the URL changes every run. The cell keeps draining logs, restarts the tunnel if it drops, and re-prints the URL.

In [ ]:
import threading, time, re, os, stat, subprocess, urllib.request

PORT = 5000
CLOUDFLARED = "./cloudflared"
_URL_RE = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")
_BANNER = "=" * 70

# If this cell was run before, stop the previous tunnel first.
try:
    tunnel.terminate()
except NameError:
    pass

def _run_flask():
    app.run(host="0.0.0.0", port=PORT, use_reloader=False)

if not any(t.name == "flask" for t in threading.enumerate()):
    threading.Thread(target=_run_flask, name="flask", daemon=True).start()
    time.sleep(1.5)  # let Flask bind before the tunnel points at it

if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
    os.chmod(CLOUDFLARED, os.stat(CLOUDFLARED).st_mode | stat.S_IEXEC)

def _start_tunnel():
    proc = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    found = {"url": None}
    def _drain():
        for line in proc.stdout:  # keep draining so cloudflared never blocks on a full pipe
            if found["url"] is None:
                m = _URL_RE.search(line)
                if m:
                    found["url"] = m.group(0)
    threading.Thread(target=_drain, daemon=True).start()
    for _ in range(40):
        if found["url"]:
            break
        time.sleep(1)
    return proc, found["url"]

tunnel, public_url = _start_tunnel()

try:
    with urllib.request.urlopen(f"http://localhost:{PORT}/api/health", timeout=5) as r:
        print("Local Flask health:", r.read().decode())
except Exception as e:
    print("!! Flask health check failed:", e)

if public_url:
    print(f"\n{_BANNER}\n  BACKEND LIVE:  {public_url}\n"
          f"  Paste this URL into the frontend's 'Connect the backend' box.\n{_BANNER}")
else:
    print("\nTunnel URL not detected -- check for a cloudflared error above, then re-run this cell.")

print("\nLeave this cell running. Re-run it (and re-paste the URL) if the frontend stops connecting.\n")

# ---- hold the cell open; relaunch the tunnel if it drops ----
try:
    _last = time.time()
    while True:
        time.sleep(15)
        if tunnel.poll() is not None:
            print("[tunnel dropped -- restarting]")
            tunnel, public_url = _start_tunnel()
            if public_url:
                print(f"\n{_BANNER}\n  NEW BACKEND URL:  {public_url}\n  Re-paste this into the frontend.\n{_BANNER}\n")
        elif time.time() - _last > 300:
            _last = time.time()
            print(f"[still live: {public_url}]")
except KeyboardInterrupt:
    print("\nStopping tunnel.")
    tunnel.terminate()

INFO:werkzeug:127.0.0.1 - - [11/Sep/2026 13:38:34] "GET /api/health HTTP/1.1" 200 -


Local Flask health: {"api_key_set":true,"groq_api_key_set":true,"has_api_key":true,"max_output_tokens":800,"model":"qwen/qwen3.8-27b","sandbox_timeout_s":10,"status":"ok"}


  BACKEND LIVE:  https://polished-hope-princess-kinase.trycloudflare.com
  Paste this URL into the frontend's 'Connect the backend' box.

Leave this cell running. Re-run it (and re-paste the URL) if the frontend stops connecting.



INFO:werkzeug:127.0.0.1 - - [11/Sep/2026 13:38:52] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Sep/2026 13:38:57] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Sep/2026 13:39:01] "OPTIONS /api/run HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Sep/2026 13:39:02] "POST /api/run HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Sep/2026 13:39:41] "OPTIONS /api/run HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Sep/2026 13:39:42] "POST /api/run HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Sep/2026 13:40:14] "OPTIONS /api/run HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Sep/2026 13:40:15] "POST /api/run HTTP/1.1" 200 -


## Known limitations of this prototype

- **Sandbox is process-isolated, not network/syscall-isolated.** A determined adversarial script could still open a socket. Fine for trusted classroom use against a code-generation model; not sufficient for untrusted multi-tenant use without OS-level controls (containers/seccomp).
- **`TRAJECTORY_LOG` and `SESSIONS` are single in-memory globals** — fine for one Colab session/demo, not for concurrent multi-user deployment.
- **Groq free-tier output rate limit (~1000 tokens/min)** — the loop makes 1-2 model calls per attempt, so a full 3-attempt task can pause 30-60s mid-run while `call_llm` backs off on a 429. Not a hang. Raise the ceiling by setting `MODEL_ID` to a higher-limit model or upgrading the Groq tier.
- **No automated correctness check** — matches the design doc's decision to keep success HITL-confirmed rather than guessing from "no traceback."
- **Streaming is per-attempt, not per-token** — the front-end updates as each attempt finishes (via NDJSON chunks over `fetch`), not while the model is still generating a single response.
- **The Cloudflare quick tunnel is public and unauthenticated, and CORS is wide open (`*`)** — anyone who has the `trycloudflare.com` link can drive the agent (and thus the sandbox) while the cell runs. Fine for a short demo; add a shared-secret header or shut the tunnel down otherwise.
- **The tunnel URL changes on every re-run** — you'll re-paste it into the UI's Settings each Colab session (it's saved in `localStorage` per browser, so only when the backend restarts).
- **The backend dies with the Colab runtime** — free Colab disconnects after ~90 min idle (12 h max), and closing the tab eventually kills it too. When the frontend says "could not reach it", the fix is almost always: reconnect Colab, re-run the launch cell (§10b), re-paste the new URL. Keeping the §10b cell running and the tab open is what holds it up.
- **Ablation harness (ReAct-only vs. ReAct+Reflexion, §9) is not built yet** — `error_type` classification is in place to support it, but running the actual comparison across the easy/medium/hard task set is the next step.